# EEG_13 — Meta-Learning per Imagined Speech
## Few-Shot Subject Adaptation con Prototypical Networks

### Motivazione
La variabilita inter-soggetto (eps2=0.85) rende i modelli cross-soggetto deboli.
L'idea meta-learning: anziche ignorare la variabilita, la usiamo come segnale di training.

**Setup**: ogni soggetto e un "task". Il meta-learner impara a classificare imagined speech
dati K esempi etichettati del soggetto target (K-shot adaptation).

### Approccio: Prototypical Networks (Snell et al. 2017)
- Encoder GNN uguale a EEG_09 (Temporal CNN + ChebConv) — senza testa di classificazione
- Prototipo: media degli embedding del support set per classe
- Classificazione: distanza euclidea nel embedding space
- Training: episodico — ogni episodio = un soggetto, N-way K-shot

### Perche non MAML
- MAML richiede gradienti del secondo ordine — oneroso
- Prototypical Networks: piu semplice, interpretabile, competitive in letteratura EEG
- Estendibile: dopo il prototipico si puo aggiungere MAML come ablation

### Schema Train/Val/Test
- Meta-train: soggetti 0-49 (50 soggetti)
- Meta-val  : soggetti 50-59 (10 soggetti)
- Meta-test : soggetti 60-69 (10 soggetti)


## Setup

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import warnings; warnings.filterwarnings('ignore')
import random, json, time
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import ChebConv, global_mean_pool

import sys
project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

META_CSV  = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH = project_root / "src" / "io" / "ebneuro.locs"
CKPT_DIR  = project_root / "data" / "interim" / "eeg13_checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES   = project_root / "figures"; FIGURES.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps"
                       if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")


## Configurazione Episodica

In [ ]:
# ============================================================
# IPER-PARAMETRI META-LEARNING
# ============================================================

LABEL_SCHEME = "concr4"      # 4 classi semantiche
N_WAY        = 4             # classi per episodio (= n classi schema)
K_SHOT       = 5             # trial support per classe
Q_QUERY      = 10            # trial query per classe per episodio
N_EPISODES_TRAIN = 500       # episodi di meta-training
N_EPISODES_VAL   = 100
N_EPISODES_TEST  = 200
META_LR      = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 30            # early stop su val_bacc

# Architettura encoder (uguale a EEG_09 ChebGCNSkip)
K_GRAPH    = 6       # k-NN per grafo PCC per-trial
CHEB_K     = 2       # ordine ChebConv
HIDDEN_DIM = 64      # dim hidden GNN
EMBED_DIM  = 128     # dim embedding (uscita encoder prima del prototipo)
TEMP_CHANNELS = [32, 64]  # canali temporal CNN

# Split soggetti
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 70))

print(f"N-way {N_WAY}-shot | Q={Q_QUERY} | episodi train={N_EPISODES_TRAIN}")
print(f"Soggetti train={len(SUBJ_TRAIN)} val={len(SUBJ_VAL)} test={len(SUBJ_TEST)}")


## Caricamento Metadata e Mapping

In [ ]:
import sys
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

meta = pd.read_csv(META_CSV)

# Filtro epoche corrotte note
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110))
)].reset_index(drop=True)

# Label scheme
labelid2cluster, cluster2name, n_clusters = load_label_scheme(LABEL_SCHEME)
assert n_clusters == N_WAY, f"N_WAY={N_WAY} != n_clusters={n_clusters}"

meta["cluster"] = meta["label_idx"].map(labelid2cluster)
meta = meta.dropna(subset=["cluster"]).copy()
meta["cluster"] = meta["cluster"].astype(int)

# Subject index: ricava da path_h5 (formato: .../XX_YY.h5)
def _subj_from_path(p):
    import re
    m = re.search(r'(\d+)_\d+\.h5', str(p))
    return int(m.group(1)) if m else -1

meta["subject_idx"] = meta["path_h5"].apply(_subj_from_path)
meta = meta[meta["subject_idx"] >= 0].reset_index(drop=True)

print(f"Totale trial: {len(meta)}")
print(f"Soggetti: {meta['subject_idx'].nunique()}")
print(f"Classi:   {meta['cluster'].nunique()}")

# Indice: subject_idx -> lista di record (dict) per trial lookup veloce
from collections import defaultdict
subj2trials = defaultdict(list)
for row in meta[["path_h5","epoch_idx","cluster","subject_idx"]].itertuples(index=False):
    subj2trials[row.subject_idx].append({
        "path_h5"  : row.path_h5,
        "epoch_idx": row.epoch_idx,
        "cluster"  : row.cluster,
    })
print(f"Soggetti indicizzati: {len(subj2trials)}")


## Funzioni Grafo per-trial

In [ ]:
# ============================================================
# FUNZIONI GRAFO — identiche a EEG_09 per coerenza
# ============================================================

def load_epoch(path_h5: str, epoch_idx: int, keep_idx):
    """Carica un trial da H5, restituisce (N_ch, T) float32."""
    with h5py.File(path_h5, "r") as f:
        x = f["eeg"][epoch_idx]          # (N_ch, T)
    return x[keep_idx].astype(np.float32)

def pcc_to_edge_index(x_np: np.ndarray, k: int) -> torch.LongTensor:
    """k-NN graph da PCC per un singolo trial. Bidirezionale."""
    C = np.corrcoef(x_np)                # (N, N)
    np.fill_diagonal(C, -1.0)
    rows, cols = [], []
    for i in range(C.shape[0]):
        top_k = np.argsort(C[i])[-k:]
        for j in top_k:
            rows += [i, j]; cols += [j, i]
    return torch.tensor([rows, cols], dtype=torch.long)

# Keep channels: rimuovi A1, A2 (idx 59, 60 in H5 a 61 canali)
_all_ch = list(range(61))
_DROP   = [59, 60]   # A1, A2
keep_idx = [c for c in _all_ch if c not in _DROP]
N_CH = len(keep_idx)
print(f"Canali: {N_CH} (drop A1/A2)")


## Episode Sampler

In [ ]:
def sample_episode(subj_idx: int,
                    n_way: int, k_shot: int, q_query: int,
                    keep_idx, k_graph: int,
                    device=DEVICE):
    """
    Campiona un episodio N-way K-shot da un singolo soggetto.

    Restituisce:
      support_batch : Batch PyG  (n_way * k_shot grafi)
      support_labels: LongTensor (n_way * k_shot,) — 0..n_way-1
      query_batch   : Batch PyG  (n_way * q_query grafi)
      query_labels  : LongTensor (n_way * q_query,)
    """
    trials_by_class = defaultdict(list)
    for t in subj2trials[subj_idx]:
        trials_by_class[t["cluster"]].append(t)

    # Filtra classi con abbastanza trial
    valid_classes = [c for c, ts in trials_by_class.items()
                     if len(ts) >= k_shot + q_query]
    if len(valid_classes) < n_way:
        return None   # episodio impossibile — skip

    chosen_classes = random.sample(valid_classes, n_way)
    class_remap = {c: i for i, c in enumerate(chosen_classes)}

    support_data, support_lbl = [], []
    query_data,   query_lbl   = [], []

    for cls in chosen_classes:
        pool = random.sample(trials_by_class[cls], k_shot + q_query)
        sup_pool, qry_pool = pool[:k_shot], pool[k_shot:]

        for t in sup_pool:
            x_np = load_epoch(t["path_h5"], t["epoch_idx"], keep_idx)
            ei   = pcc_to_edge_index(x_np, k_graph)
            x_t  = torch.tensor(x_np, dtype=torch.float32)
            support_data.append(Data(x=x_t, edge_index=ei))
            support_lbl.append(class_remap[cls])

        for t in qry_pool:
            x_np = load_epoch(t["path_h5"], t["epoch_idx"], keep_idx)
            ei   = pcc_to_edge_index(x_np, k_graph)
            x_t  = torch.tensor(x_np, dtype=torch.float32)
            query_data.append(Data(x=x_t, edge_index=ei))
            query_lbl.append(class_remap[cls])

    sup_batch = Batch.from_data_list(support_data).to(device)
    qry_batch = Batch.from_data_list(query_data).to(device)
    sup_lbl   = torch.tensor(support_lbl, dtype=torch.long, device=device)
    qry_lbl   = torch.tensor(query_lbl,   dtype=torch.long, device=device)

    return sup_batch, sup_lbl, qry_batch, qry_lbl


# Test sampler
ep = sample_episode(SUBJ_TRAIN[0], N_WAY, K_SHOT, Q_QUERY, keep_idx, K_GRAPH)
if ep:
    sb, sl, qb, ql = ep
    print(f"Support batch: {sb.num_graphs} grafi, x={sb.x.shape}")
    print(f"Query   batch: {qb.num_graphs} grafi, x={qb.x.shape}")
    print(f"Support labels: {sl.tolist()}")
else:
    print("Episodio non campionabile — verifica i dati")


## Architettura: Prototypical GNN Encoder

In [ ]:
class TemporalEncoder(nn.Module):
    """CNN 1D per-nodo: (N, T) -> (N, hidden_dim)."""
    def __init__(self, n_times: int, out_dim: int, channels=(32, 64)):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, channels[0], kernel_size=25, padding=12),
            nn.BatchNorm1d(channels[0]), nn.ELU(),
            nn.Conv1d(channels[0], channels[1], kernel_size=10, padding=5),
            nn.BatchNorm1d(channels[1]), nn.ELU(),
            nn.AdaptiveAvgPool1d(out_dim // channels[1]),
        )
        self.fc = nn.Linear(channels[1] * (out_dim // channels[1]), out_dim)

    def forward(self, x):
        # x: (N_nodes_batch, T)
        h = self.conv(x.unsqueeze(1))   # (N, C, L)
        h = h.flatten(1)                # (N, C*L)
        return self.fc(h)               # (N, out_dim)


class ProtoGNNEncoder(nn.Module):
    """
    Temporal CNN + ChebConv + global_mean_pool -> embedding di dimensione embed_dim.
    Nessuna testa di classificazione — solo embedding per prototipo.
    """
    def __init__(self, n_times: int, hidden_dim: int, embed_dim: int,
                 cheb_k: int = 2, temp_channels=(32, 64)):
        super().__init__()
        self.temp = TemporalEncoder(n_times, hidden_dim, temp_channels)
        self.conv1 = ChebConv(hidden_dim, hidden_dim, K=cheb_k)
        self.conv2 = ChebConv(hidden_dim, embed_dim,  K=cheb_k)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(embed_dim)
        self.drop  = nn.Dropout(0.3)

        # Skip connection
        self.skip  = nn.Linear(hidden_dim, embed_dim)

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        h  = self.temp(x)                          # (N, hidden)
        h1 = F.elu(self.bn1(self.conv1(h,  ei)))   # (N, hidden)
        h1 = self.drop(h1)
        h2 = F.elu(self.bn2(self.conv2(h1, ei)))   # (N, embed)
        h2 = h2 + self.skip(h)                     # skip
        z  = global_mean_pool(h2, batch)            # (B, embed)
        return F.normalize(z, dim=-1)               # unit norm


# Istanzia
N_TIMES = 384   # campioni temporali per trial (256 Hz * 1.5s)
encoder = ProtoGNNEncoder(
    n_times=N_TIMES,
    hidden_dim=HIDDEN_DIM,
    embed_dim=EMBED_DIM,
    cheb_k=CHEB_K,
    temp_channels=TEMP_CHANNELS,
).to(DEVICE)

n_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f"Parametri encoder: {n_params:,}")

# Sanity: forward su un episodio
with torch.no_grad():
    sb, sl, qb, ql = ep
    z = encoder(sb)
    print(f"Embedding support: {z.shape} (atteso [{N_WAY*K_SHOT}, {EMBED_DIM}])")


## Prototypical Loss

In [ ]:
def prototypical_loss(encoder, sup_batch, sup_lbl, qry_batch, qry_lbl, n_way):
    """
    Prototypical loss episodica (Snell et al. 2017).
    1. Codifica support e query
    2. Calcola prototipi = mean degli embedding per classe
    3. Classifica query via distanza euclidea ai prototipi
    4. Restituisce (loss, predicted_labels)
    """
    z_sup = encoder(sup_batch)   # (n_way*k_shot, D)
    z_qry = encoder(qry_batch)  # (n_way*q_query, D)

    # Prototipi: media per classe
    protos = torch.stack([
        z_sup[sup_lbl == c].mean(0) for c in range(n_way)
    ])  # (n_way, D)

    # Distanze euclidee (quadrate)
    dists = torch.cdist(z_qry, protos)   # (Q, n_way)

    # Log-probabilita: -distanza (piu vicino = piu probabile)
    log_p  = F.log_softmax(-dists, dim=1)
    loss   = F.nll_loss(log_p, qry_lbl)
    preds  = log_p.argmax(dim=1)

    return loss, preds


# Test loss
with torch.no_grad():
    loss_t, preds_t = prototypical_loss(encoder, sb, sl, qb, ql, N_WAY)
    bacc_t = balanced_accuracy_score(ql.cpu(), preds_t.cpu())
    print(f"Loss test: {loss_t.item():.4f}  |  bAcc test: {bacc_t:.3f}  (chance={1/N_WAY:.3f})")


## Training Loop Episodico

In [ ]:
optimizer = torch.optim.Adam(encoder.parameters(), lr=META_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPISODES_TRAIN)

def run_split(subj_list, n_episodes, train=True):
    """Esegui n_episodes su subj_list. Ritorna (mean_loss, mean_bacc)."""
    losses, baccs = [], []
    for _ in range(n_episodes):
        # Campiona soggetto casuale
        subj = random.choice(subj_list)
        ep   = sample_episode(subj, N_WAY, K_SHOT, Q_QUERY, keep_idx, K_GRAPH)
        if ep is None: continue
        sb, sl, qb, ql = ep

        if train:
            optimizer.zero_grad()
            loss, preds = prototypical_loss(encoder, sb, sl, qb, ql, N_WAY)
            loss.backward()
            nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
            optimizer.step()
        else:
            with torch.no_grad():
                loss, preds = prototypical_loss(encoder, sb, sl, qb, ql, N_WAY)

        losses.append(loss.item())
        baccs.append(balanced_accuracy_score(ql.cpu(), preds.cpu()))

    return np.mean(losses), np.mean(baccs)


history = {"train_loss":[], "train_bacc":[], "val_loss":[], "val_bacc":[]}
best_val_bacc = 0.0
no_improve    = 0
best_ckpt     = CKPT_DIR / "proto_gnn_best.pt"

print(f"Inizio meta-training ({N_EPISODES_TRAIN} episodi train, {N_EPISODES_VAL} val)")
print(f"Chance level: {1/N_WAY:.3f}")
print(f"{'Ep':>4}  {'TrLoss':>8}  {'TrBAcc':>8}  {'VaLoss':>8}  {'VaBAcc':>8}")

for epoch in range(1, 101):   # 100 meta-epoch, ogni epoch = 5 episodi train + 1 val
    tr_loss, tr_bacc = run_split(SUBJ_TRAIN, n_episodes=5, train=True)
    scheduler.step()

    if epoch % 5 == 0:
        encoder.eval()
        va_loss, va_bacc = run_split(SUBJ_VAL, n_episodes=20, train=False)
        encoder.train()

        history["train_loss"].append(tr_loss)
        history["train_bacc"].append(tr_bacc)
        history["val_loss"].append(va_loss)
        history["val_bacc"].append(va_bacc)

        print(f"{epoch:4d}  {tr_loss:8.4f}  {tr_bacc:8.3f}  {va_loss:8.4f}  {va_bacc:8.3f}")

        if va_bacc > best_val_bacc:
            best_val_bacc = va_bacc
            no_improve    = 0
            torch.save(encoder.state_dict(), best_ckpt)
        else:
            no_improve += 1
            if no_improve >= PATIENCE // 5:
                print(f"Early stop a epoch {epoch}")
                break

print(f"\nMigliore val_bacc: {best_val_bacc:.4f}")


## Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep_idx = list(range(5, len(history["train_loss"])*5+1, 5))

ax1.plot(ep_idx, history["train_loss"], label="train"); ax1.plot(ep_idx, history["val_loss"], label="val")
ax1.set_xlabel("Meta-epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Prototypical Loss"); ax1.legend()

ax2.plot(ep_idx, history["train_bacc"], label="train"); ax2.plot(ep_idx, history["val_bacc"], label="val")
ax2.axhline(1/N_WAY, ls="--", color="gray", label="chance")
ax2.set_xlabel("Meta-epoch"); ax2.set_ylabel("bAcc"); ax2.set_title("Balanced Accuracy"); ax2.legend()

fig.tight_layout()
fig.savefig(FIGURES / "eeg13_proto_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## Valutazione Meta-Test (soggetti mai visti)

In [ ]:
# Carica best checkpoint
encoder.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
encoder.eval()

print("=== META-TEST (soggetti 60-69) ===")
te_loss, te_bacc = run_split(SUBJ_TEST, n_episodes=N_EPISODES_TEST, train=False)
print(f"Test loss : {te_loss:.4f}")
print(f"Test bAcc : {te_bacc:.4f}  (chance = {1/N_WAY:.3f})")
print(f"Lift      : {te_bacc / (1/N_WAY):.2f}x chance")

# Per-soggetto breakdown
print("\n--- Per soggetto ---")
per_subj = {}
for subj in SUBJ_TEST:
    _, s_bacc = run_split([subj], n_episodes=50, train=False)
    per_subj[subj] = s_bacc
    print(f"  Soggetto {subj:3d}: {s_bacc:.3f}")

# Plot per-soggetto
fig, ax = plt.subplots(figsize=(10, 4))
subjs = list(per_subj.keys())
baccs = [per_subj[s] for s in subjs]
ax.bar(subjs, baccs, color=["steelblue" if b > 1/N_WAY else "tomato" for b in baccs])
ax.axhline(1/N_WAY, ls="--", color="gray", label="chance")
ax.set_xlabel("Soggetto"); ax.set_ylabel("bAcc"); ax.set_title(f"Meta-Test per soggetto ({N_WAY}-way {K_SHOT}-shot)")
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES / "eeg13_proto_per_subject_test.png", dpi=150, bbox_inches="tight")
plt.show()


## Visualizzazione Embedding Space (PCA)

In [ ]:
# Raccoglie embedding di tutti i trial di un soggetto test
# per visualizzare se le classi si separano nello spazio prototipico

VIZ_SUBJ = SUBJ_TEST[0]
trials_viz = subj2trials[VIZ_SUBJ]

embeddings, labels_viz = [], []
encoder.eval()
with torch.no_grad():
    for t in trials_viz[:200]:   # max 200 trial per velocita
        try:
            x_np = load_epoch(t["path_h5"], t["epoch_idx"], keep_idx)
            ei   = pcc_to_edge_index(x_np, K_GRAPH)
            x_t  = torch.tensor(x_np, dtype=torch.float32).to(DEVICE)
            d    = Data(x=x_t, edge_index=ei.to(DEVICE))
            b    = Batch.from_data_list([d]).to(DEVICE)
            z    = encoder(b).squeeze(0).cpu().numpy()
            embeddings.append(z)
            labels_viz.append(t["cluster"])
        except Exception:
            continue

emb = np.array(embeddings)
lbl = np.array(labels_viz)
pca = PCA(n_components=2)
emb2d = pca.fit_transform(emb)

fig, ax = plt.subplots(figsize=(8, 6))
pal = plt.cm.tab10(np.linspace(0, 0.4, N_WAY))
for c in range(N_WAY):
    mask = lbl == c
    ax.scatter(emb2d[mask, 0], emb2d[mask, 1],
               alpha=0.6, s=30, color=pal[c], label=f"Classe {c}")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"Embedding space — soggetto {VIZ_SUBJ} (PCA)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "eeg13_proto_embedding_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Varianza spiegata: {pca.explained_variance_ratio_.sum()*100:.1f}%")


## Ablation: Effetto K-Shot

Quanti esempi labeled servono per adattarsi a un nuovo soggetto?

In [ ]:
K_SHOT_VALUES = [1, 2, 5, 10, 20]
ablation_results = {}

encoder.eval()
for k in K_SHOT_VALUES:
    # Ridefinisce un sampler temporaneo con k diverso
    _baccs = []
    for _ in range(100):
        subj = random.choice(SUBJ_TEST)
        ep   = sample_episode(subj, N_WAY, k, Q_QUERY, keep_idx, K_GRAPH)
        if ep is None: continue
        sb, sl, qb, ql = ep
        with torch.no_grad():
            _, preds = prototypical_loss(encoder, sb, sl, qb, ql, N_WAY)
        _baccs.append(balanced_accuracy_score(ql.cpu(), preds.cpu()))
    ablation_results[k] = np.mean(_baccs)
    print(f"k={k:3d}:  bAcc={ablation_results[k]:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K_SHOT_VALUES, [ablation_results[k] for k in K_SHOT_VALUES], marker='o', color='steelblue')
ax.axhline(1/N_WAY, ls='--', color='gray', label='chance')
ax.set_xlabel('K-shot (n. esempi labeled per classe)'); ax.set_ylabel('bAcc')
ax.set_title('Ablation: effetto K-shot sul soggetto nuovo')
ax.legend(); fig.tight_layout()
fig.savefig(FIGURES / "eeg13_proto_kshot_ablation.png", dpi=150, bbox_inches="tight")
plt.show()


## Note e Prossimi Passi

### Cosa fa questo notebook
- Meta-training episodico su 50 soggetti con Prototypical Networks
- Encoder GNN (Temporal CNN + ChebConv) — stesso di EEG_09, senza testa classificazione
- Valutazione su 10 soggetti mai visti in training
- Ablation K-shot: quanti esempi labeled servono per il soggetto nuovo

### Interpretazione risultati
- Se `test_bacc > chance (0.25)` con K=5: il meta-learner trasferisce conoscenza
- Se la curva K-shot sale rapidamente: pochi esempi bastano (desiderato)
- Visualizzazione PCA: se le classi si separano nell'embedding → il GNN cattura struttura semantica

### Sviluppi possibili
1. **MAML** come confronto: inner loop per-soggetto, outer loop meta, 1st-order approx (FOMAML)
2. **Conditional Prototypes**: usa meta-dati del soggetto (es. cluster EEG_08) per condizionare i prototipi
3. **Subject-conditioned encoder**: aggiunge embedding soggetto (da EEG_08 cluster) come input
4. **Cross-modal**: usa i prototipi EEG per allinearsi a word embeddings semantici (contributo originale)

### Riferimenti
- Snell et al. 2017 — Prototypical Networks for Few-shot Learning
- Zhao et al. 2023 — Thinking Race: few-shot imagined speech decoding
- Finn et al. 2017 — MAML (per l'ablation futura)
